cell 1: kiểm tra đường dẫn

In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/taranvee/smart-home-dataset-with-weather-information/HomeC.csv


cell 2: đọc dữ liệu kiểm tra cột

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("/kaggle/input/datasets/taranvee/smart-home-dataset-with-weather-information/HomeC.csv")
print(df.shape)
print(df.columns.tolist())
df.head()


/tmp/ipykernel_58/4058704858.py:4: DtypeWarning: Columns (0,27) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/kaggle/input/datasets/taranvee/smart-home-dataset-with-weather-information/HomeC.csv")


(503911, 32)
['time', 'use [kW]', 'gen [kW]', 'House overall [kW]', 'Dishwasher [kW]', 'Furnace 1 [kW]', 'Furnace 2 [kW]', 'Home office [kW]', 'Fridge [kW]', 'Wine cellar [kW]', 'Garage door [kW]', 'Kitchen 12 [kW]', 'Kitchen 14 [kW]', 'Kitchen 38 [kW]', 'Barn [kW]', 'Well [kW]', 'Microwave [kW]', 'Living room [kW]', 'Solar [kW]', 'temperature', 'icon', 'humidity', 'visibility', 'summary', 'apparentTemperature', 'pressure', 'windSpeed', 'cloudCover', 'windBearing', 'precipIntensity', 'dewPoint', 'precipProbability']


,time,use [kW],gen [kW],House overall [kW],Dishwasher [kW],Furnace 1 [kW],Furnace 2 [kW],Home office [kW],Fridge [kW],Wine cellar [kW],...,visibility,summary,apparentTemperature,pressure,windSpeed,cloudCover,windBearing,precipIntensity,dewPoint,precipProbability
0,1451624400,0.932833,0.003483,0.932833,0.000033,0.020700,0.061917,0.442633,0.124150,0.006983,...,10.0,Clear,29.26,1016.91,9.18,cloudCover,282.0,0.0,24.4,0.0
1,1451624401,0.934333,0.003467,0.934333,0.000000,0.020717,0.063817,0.444067,0.124000,0.006983,...,10.0,Clear,29.26,1016.91,9.18,cloudCover,282.0,0.0,24.4,0.0
2,1451624402,0.931817,0.003467,0.931817,0.000017,0.020700,0.062317,0.446067,0.123533,0.006983,...,10.0,Clear,29.26,1016.91,9.18,cloudCover,282.0,0.0,24.4,0.0
3,1451624403,1.022050,0.003483,1.022050,0.000017,0.106900,0.068517,0.446583,0.123133,0.006983,...,10.0,Clear,29.26,1016.91,9.18,cloudCover,282.0,0.0,24.4,0.0
4,1451624404,1.139400,0.003467,1.139400,0.000133,0.236933,0.063983,0.446533,0.122850,0.006850,...,10.0,Clear,29.26,1016.91,9.18,cloudCover,282.0,0.0,24.4,0.0


Cell 3 xử lý thời gian và chọn cột cần thiết

In [ ]:
start_time = pd.Timestamp("2016-01-01 00:00:00")
df["timestamp"] = start_time + pd.to_timedelta(np.arange(len(df)), unit="min")

df = df.rename(columns={"use [kW]": "consumption_kw"})

keep_cols = ["timestamp", "consumption_kw", "temperature", "humidity"]
df = df[keep_cols].sort_values("timestamp").reset_index(drop=True)
print("Khoảng thời gian sau khi dựng lại:", df["timestamp"].min(), "→", df["timestamp"].max())
print(df.shape)
df.head()

Khoảng thời gian sau khi dựng lại: 2016-01-01 00:00:00 → 2016-12-15 22:30:00
(503911, 4)


,timestamp,consumption_kw,temperature,humidity
0,2016-01-01 00:00:00,0.932833,36.14,0.62
1,2016-01-01 00:01:00,0.934333,36.14,0.62
2,2016-01-01 00:02:00,0.931817,36.14,0.62
3,2016-01-01 00:03:00,1.022050,36.14,0.62
4,2016-01-01 00:04:00,1.139400,36.14,0.62


Cell 4: Resample về 15p/ điểm khớp với Model 1

In [4]:
df = df.set_index("timestamp")
df_15min = df.resample("15min").mean().dropna().reset_index()
print("Số dòng sau resample 15 phút:", len(df_15min))
df_15min.head()

Số dòng sau resample 15 phút: 33594


,timestamp,consumption_kw,temperature,humidity
0,2016-01-01 00:00:00,1.240810,36.140,0.620000
1,2016-01-01 00:15:00,1.013110,36.140,0.620000
2,2016-01-01 00:30:00,1.063666,36.140,0.620000
3,2016-01-01 00:45:00,0.858933,36.104,0.618667
4,2016-01-01 01:00:00,0.980608,35.870,0.610000


cell 5: Làm sạch dữ liệu

In [5]:
def clean_consumption_data(df):
    df = df.copy()
    df = df.drop_duplicates(subset="timestamp")
    df = df.sort_values("timestamp").reset_index(drop=True)
    df["consumption_kw"] = df["consumption_kw"].interpolate(method="linear")
    df["temperature"] = df["temperature"].interpolate(method="linear")
    df["humidity"] = df["humidity"].interpolate(method="linear")
    df = df[df["consumption_kw"] >= 0]
    return df

df_clean = clean_consumption_data(df_15min)
print(df_clean.isna().sum())

timestamp         0
consumption_kw    0
temperature       0
humidity          0
dtype: int64


cell 6, tạo đặc trưng

In [ ]:
def create_consumption_features(df):
    df = df.copy()
    hour = df["timestamp"].dt.hour + df["timestamp"].dt.minute / 60
    df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
 
    steps_per_day = 24 * 60 // 15
    df["consumption_lag_24h"] = df["consumption_kw"].shift(steps_per_day)
    df["consumption_lag_1"] = df["consumption_kw"].shift(1)
 
    df = df.dropna().reset_index(drop=True)
    return df
 
df_features_c = create_consumption_features(df_clean)
print(df_features_c.shape)
df_features_c.head()

(33498, 8)


,timestamp,consumption_kw,temperature,humidity,hour_sin,hour_cos,consumption_lag_24h,consumption_lag_1
0,2016-01-02 00:00:00,1.449704,29.880,0.600000,0.000000,1.000000,1.240810,2.759143
1,2016-01-02 00:15:00,1.290899,29.560,0.600000,0.065403,0.997859,1.013110,1.449704
2,2016-01-02 00:30:00,0.816346,29.560,0.600000,0.130526,0.991445,1.063666,1.290899
3,2016-01-02 00:45:00,1.840657,29.560,0.600000,0.195090,0.980785,0.858933,0.816346
4,2016-01-02 01:00:00,1.153761,29.224,0.609333,0.258819,0.965926,0.980608,1.840657


In [7]:
feature_cols_c = ["temperature", "humidity", "hour_sin", "hour_cos", "consumption_kw",
                   "consumption_lag_1", "consumption_lag_24h"]
target_col_c = "consumption_kw"
 
missing = [c for c in feature_cols_c if c not in df_features_c.columns]
assert not missing, f"Thiếu cột: {missing}"
print("feature_cols_c:", feature_cols_c)
print("target_col_c:", target_col_c)
print("Số dòng sẵn sàng:", len(df_features_c))

feature_cols_c: ['temperature', 'humidity', 'hour_sin', 'hour_cos', 'consumption_kw', 'consumption_lag_1', 'consumption_lag_24h']
target_col_c: consumption_kw
Số dòng sẵn sàng: 33498


Cell 8; tách train/test

In [8]:
def split_train_test(df, test_ratio=0.2):
    split_idx = int(len(df) * (1 - test_ratio))
    return df.iloc[:split_idx].reset_index(drop=True), df.iloc[split_idx:].reset_index(drop=True)

train_df_c, test_df_c = split_train_test(df_features_c)
print(f"Train: {len(train_df_c)} dòng | Test: {len(test_df_c)} dòng")

Train: 26798 dòng | Test: 6700 dòng


Cell 9

In [9]:
from sklearn.preprocessing import MinMaxScaler

scaler_X_c = MinMaxScaler()
scaler_y_c = MinMaxScaler()

X_train_scaled_c = scaler_X_c.fit_transform(train_df_c[feature_cols_c])
X_test_scaled_c = scaler_X_c.transform(test_df_c[feature_cols_c])

y_train_scaled_c = scaler_y_c.fit_transform(train_df_c[[target_col_c]])
y_test_scaled_c = scaler_y_c.transform(test_df_c[[target_col_c]])

def create_sequences(X, y, window_size=96):
    n_sequences = len(X) - window_size
    if n_sequences <= 0:
        raise ValueError(
            f"Không đủ dữ liệu để tạo sequence: chỉ có {len(X)} dòng nhưng "
            f"cần hơn {window_size} dòng (window_size). Kiểm tra lại bước "
            f"resample/dropna phía trên, có thể đã loại bỏ nhầm quá nhiều dữ liệu."
        )
    X_seq, y_seq = [], []
    for i in range(n_sequences):
        X_seq.append(X[i: i + window_size])
        y_seq.append(y[i + window_size])
    return np.array(X_seq), np.array(y_seq)

WINDOW = 96
X_train_seq_c, y_train_seq_c = create_sequences(X_train_scaled_c, y_train_scaled_c, WINDOW)
X_test_seq_c, y_test_seq_c = create_sequences(X_test_scaled_c, y_test_scaled_c, WINDOW)

print("Train shape:", X_train_seq_c.shape)
print("Test shape:", X_test_seq_c.shape)

Train shape: (26702, 96, 7)
Test shape: (6604, 96, 7)


Cell 10: Train LSTM

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error, mean_squared_error

model_c = Sequential([
    LSTM(64, activation="tanh", input_shape=(WINDOW, len(feature_cols_c)), return_sequences=False),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)
])

model_c.compile(optimizer="adam", loss="mse")
model_c.summary()

early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history_c = model_c.fit(
    X_train_seq_c, y_train_seq_c,
    validation_data=(X_test_seq_c, y_test_seq_c),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

y_pred_scaled_c = model_c.predict(X_test_seq_c)
y_pred_c = scaler_y_c.inverse_transform(y_pred_scaled_c)
y_test_actual_c = scaler_y_c.inverse_transform(y_test_seq_c)

mae_c = mean_absolute_error(y_test_actual_c, y_pred_c)
rmse_c = np.sqrt(mean_squared_error(y_test_actual_c, y_pred_c))
print(f"Consumption LSTM - MAE: {mae_c:.4f} kW | RMSE: {rmse_c:.4f} kW")

I0000 00:00:1790536576.451716      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790536576.454792      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,545 (80.25 KB)

 Trainable params: 20,545 (80.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 11s 9ms/step - loss: 0.0038 - val_loss: 0.0013
Epoch 2/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0030 - val_loss: 0.0011
Epoch 3/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0028 - val_loss: 0.0014
Epoch 4/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 0.0027 - val_loss: 0.0011
Epoch 5/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0027 - val_loss: 0.0012
Epoch 6/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0026 - val_loss: 0.0011
Epoch 7/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 0.0026 - val_loss: 0.0010
Epoch 8/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0026 - val_loss: 0.0010
Epoch 9/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 0.0026 - val_loss: 0.0010
Epoch 10/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 0.0026 - val_loss: 0.0011
Epoch 11/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0025 - val_loss: 0.0010
Epoch 12/100
835/835 ━━━━━━━━━━━━━━━━━━━

In [11]:
naive_pred = test_df_c[target_col_c].values[WINDOW - 1 : WINDOW - 1 + len(y_test_actual_c)]
naive_actual = y_test_actual_c.flatten()

mae_naive = mean_absolute_error(naive_actual, naive_pred)
rmse_naive = np.sqrt(mean_squared_error(naive_actual, naive_pred))

print("=== SO SANH ===")
print("Naive MAE:", round(mae_naive, 4))
print("Naive RMSE:", round(rmse_naive, 4))
print("LSTM MAE:", round(mae_c, 4))
print("LSTM RMSE:", round(rmse_c, 4))

=== SO SANH ===
Naive MAE: 0.2177
Naive RMSE: 0.3593
LSTM MAE: 0.2065
LSTM RMSE: 0.3312


Cell 11

In [ ]:
import json

sample_cons = df_clean.tail(192).copy()
sample_cons["timestamp"] = sample_cons["timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%S")

cons_points = sample_cons[["timestamp", "temperature", "humidity", "consumption_kw"]].to_dict(orient="records")

with open("/kaggle/working/sample_consumption_points.json", "w") as f:
    json.dump(cons_points, f)

print("Đã xuất", len(cons_points), "điểm consumption")

Đã xuất 192 điểm consumption


In [ ]:
import json
import joblib

model_c.save("/kaggle/working/lstm_consumption_forecast.h5")
joblib.dump(scaler_X_c, "/kaggle/working/scaler_X_consumption.pkl")
joblib.dump(scaler_y_c, "/kaggle/working/scaler_y_consumption.pkl")

reference_load_kw = float(train_df_c["consumption_kw"].median())
print("reference_load_kw (median tiêu thụ hộ HomeC lúc train):", reference_load_kw)

HOUSE_BASE_LOAD_W = {"H01": 1500, "H02": 1800, "H03": 3200, "H04": 2600}

model_config_c = {
    "feature_cols": feature_cols_c,
    "window": WINDOW,
    "slot_duration_sec": 900,
    "target": "consumption_kw (kW thật của hộ HomeC, qua scaler_y_c để inverse_transform)",
    "reference_load_kw": reference_load_kw,
    "house_base_load_w": HOUSE_BASE_LOAD_W,
    "metrics": {"lstm_mae_kw": float(mae_c), "naive_mae_kw": float(mae_naive)},
    "limitation": "Dùng chung 1 pattern hộ gia đình thật (HomeC) cho cả 4 hộ "
                  "H01-H04 (không có dữ liệu tiêu thụ thật riêng cho từng "
                  "hộ) — chỉ scale theo loadW của từng hộ trong .env, không "
                  "phản ánh khác biệt hành vi tiêu thụ thật giữa các hộ; "
                  "LSTM chỉ nhỉnh hơn naive baseline ~2.5% MAE.",
}
with open("/kaggle/working/model_config_consumption.json", "w") as f:
    json.dump(model_config_c, f, indent=2, ensure_ascii=False)

print("Đã lưu lstm_consumption_forecast.h5, scaler_X_consumption.pkl, "
      "scaler_y_consumption.pkl, model_config_consumption.json")

reference_load_kw (median tiêu thụ hộ HomeC lúc train): 0.6164733333333333
Đã lưu lstm_consumption_forecast.h5, scaler_X_consumption.pkl, scaler_y_consumption.pkl, model_config_consumption.json
